# DPR processor example with Prefect+Dask

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-520

See the associated:

  * Python module: [dpr_processor_example.py](./dpr_processor_example.py)
  * YAML file: [dpr_processor_example.yaml](./dpr_processor_example.yaml)

**NOTE: This notebook is not run from the ci/cd because we have random errors with eopf.**

## Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
image = 41e8ad0cce974322aec607c3f3cbe240
Get existing dask cluster: '41e8ad0cce974322aec607c3f3cbe240'
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/41e8ad0cce974322aec607c3f3cbe240/status
Dask workers for 'dask-eopf' are up: 2/2


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
# Other imports
import os
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# Test the DPR processing with n dummy products
output_count = 3
s3_basename = "new_zarr_product_"
s3_filenames = [f"{s3_basename}{i}" for i in range(output_count)]

# Data to test the example flow
s3_path = os.path.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "zarr"
)
my_data = {"s3_folder": s3_path, "s3_filenames": s3_filenames}

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

print(f"Output zarr products will be written to: {s3_path}")

Output zarr products will be written to: s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/zarr


In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Set environment variables for the client and dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"
dask_client.run(set_dask_env)
os.environ["HELLO_FROM"] = "client"

In [5]:
# NOTE: we need to create the S3 folder with a dummy empty file before running DPR
await s3_upload_empty_file(f"{s3_path}/.empty")

10:34:55.675 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpqf4l381_' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/zarr/.empty'.

'prefect-share/users/jgaucher/zarr/.empty'

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [6]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [7]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./dpr_processor_example.yaml"

10:34:58.067 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.war

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'dpr-flow/sprint20-dpr-example' successfully created with id      │
│ '763af68d-2162-48e7-a9e4-40094474a575'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/763af68d-2162-48e7-a9e4-40094474a575


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'dpr-flow/sprint20-dpr-example'



In [8]:
deploy_name = "dpr-flow/sprint20-dpr-example"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'dpr-flow/sprint20-dpr-example'


## Run Prefect flow

In [9]:
print(f"Remove existing zarr products from: {s3_path!r}")
s3_delete(f"{s3_path}/{s3_basename}")

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/zarr'


In [10]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'dpr-flow/sprint20-dpr-example'...
Created flow run 'nickel-crow'.
└── UUID: b910ffe7-1aac-4ae0-8cc1-96fcba446008
└── Parameters: {'s3_folder': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/zarr', 's3_filenames': ['new_zarr_product_0', 'new_zarr_product_1', 'new_zarr_product_2']}
└── Job Variables: {}
└── Scheduled start time: 2025-05-22 10:35:03 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/b910ffe7-1aac-4ae0-8cc1-96fcba446008
Watching flow run 'nickel-crow'...


10:35:09.454 | INFO    | prefect - Flow run is in state 'Pending'
10:35:13.342 | INFO    | prefect - Flow run is in state 'Running'
10:35:18.670 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


NOTE: we could also call the Prefect flow from Python code.
This is useful to debug or see the generated HTML representation.

In [11]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import dpr_processor_example
    reload(dpr_processor_example)
    
    # Run the flow
    results = dpr_processor_example.dpr_flow(**my_data)
    
    # Display HTML representation
    import IPython
    for result in results:
        display(IPython.display.HTML(result))
    del results

## Check results

In [12]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await s3_download_dir(f"{s3_path}", local_dir)
!ls -al "$local_dir"

total 24
drwxr-xr-x 5 jovyan users 4096 May 22 10:35 .
drwxrwxrwt 1 root   root  4096 May 22 10:35 ..
-rw-r--r-- 1 jovyan users    5 May 22 10:35 .empty
drwxr-xr-x 3 jovyan users 4096 May 22 10:35 new_zarr_product_0.zarr
drwxr-xr-x 3 jovyan users 4096 May 22 10:35 new_zarr_product_1.zarr
drwxr-xr-x 3 jovyan users 4096 May 22 10:35 new_zarr_product_2.zarr


In [13]:
# Open them with the zarr python package
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
!pip install zarr
import zarr

for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 44.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_0.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[38, 49, 51, ...,  7, 61, 27],
       [19, 65, 60, ..., 80, 27, 42],
       [81, 54, 19, ..., 75, 98, 74],
       ...,
       [92, 47, 32, ..., 19, 81, 46],
       [94, 45, 51, ..., 42, 64, 17],
       [75, 36,  6, ..., 71,  0, 62]], shape=(1024, 1024))

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_1.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[96, 43, 80, ..., 30, 45, 42],
       [55, 53, 36, ..., 27, 21, 50],
       [63, 28, 84, ..., 19, 93, 84],
       ...,
       [47, 35, 38, ..., 82,  3, 56],
       [55, 36, 44, ..., 17, 14, 56],
       [17,  8, 89, ..., 12, 27, 65]], shape=(1024, 1024))

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_2.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[ 2, 25, 78, ..., 83,  0, 64],
       [91, 40, 28, ..., 99, 80, 32],
       [97, 26, 70, ..., 87, 21, 11],
       ...,
       [56, 94, 47, ...,  1, 50, 54],
       [32, 42, 56, ..., 64, 49, 21],
       [39,  1, 14, ..., 86, 42, 55]], shape=(1024, 1024))

## 3. Shutdown the dask clusters

In [14]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.